In [1]:
import sys
BASE_DIR = "../../../.."
sys.path.insert(0, BASE_DIR)

import pandas as pd
import numpy as np
import ast
import random
import os
import torch
import json
from time import time
import gc
import chromadb
from tqdm import tqdm
from dataclasses import dataclass, field
from sentence_transformers import SentenceTransformer
from typing import Dict, List
from dataclasses import dataclass

random.seed(42)

from src.agents.hosted import CustomAgent
from src.utils import ReaderMetrics
from src.utils.inference_metrics import compute_predictive_entropy

QA_DATASET_PATH = "../../../../data/squadv2/qa_dataset.csv"
AGENT_MODEL_PATH = "../../../../models/Undi95/Meta-Llama-3-8B-Instruct-hf"

/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
PARAMS = {
    'version': "1",
    'num_samples': 2000,
    'model': AGENT_MODEL_PATH,
    'system_prompt': "You are an AI assistant who helps solve user issues.",
    "item_format": "- {document}",
    "user_prompt": 'Generate an answer to the given question. If you do not have enough knowledge to generate correct answer based on them, then generate the following text: "I do not have an answer to your question". Generate answer in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.',
    "prompt_format":"{user_p}\n\nQuestion:\n{q}\n\nAnswer:\n",
    'gen_strat': {'max_new_tokens': 1024, 'do_sample': False, 'num_beams': 1},
    'stub_answer': "I do not have an answer to your question",
    'calculate_entropy': True
}


METADATA_SAVE_NAME = 'metadata.json'
USER_PROPMTS_SAVE_NAME = 'user_prompts.json'
PARAMS_SAVE_NAME = 'hyperp.json'
GEN_ANSW_SAVE_NAME = 'generation_info.json'
SCORES_SAVE_NAME = 'scores.json'
LOGS_SAVE_DIR = './logs_v2'
META_INFO_DIR_NAME = 'gen_metainfo'

if os.path.exists(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}'):
    print("Dir exists")
else:
    print("Creating Dir...")
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}')
    os.mkdir(f'{LOGS_SAVE_DIR}/v{PARAMS["version"]}/{META_INFO_DIR_NAME}')

Creating Dir...


### Подключение к агенту

In [3]:
agent = CustomAgent(PARAMS['model'], output_logits=PARAMS['calculate_entropy'], use_cache=True, output_attentions=False, output_scores=False, output_hidden_states=False)
output = agent.generate(user_prompt="what is wrong with humanity?", system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])
print(output[0])

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

What a profound and complex question! As an AI assistant, I'll provide some insights and perspectives, but I must emphasize that humanity is a diverse and multifaceted entity, and there is no single answer to this question.

That being said, here are some potential issues that have been identified by experts, researchers, and individuals:

1. **Conflict and violence**: Wars, terrorism, and other forms of violence have plagued human history, causing immense suffering and destruction.
2. **Inequality and social injustice**: Systemic inequalities, discrimination, and social injustices persist, affecting marginalized groups, such as women, minorities, and the poor.
3. **Environmental degradation**: Human activities have led to significant environmental damage, including climate change, pollution, and loss of biodiversity.
4. **Mental health and well-being**: Many people struggle with mental health issues, such as depression, anxiety, and trauma, which can have a profound impact on individu

### Готовим промпт

In [4]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [5]:
USER_PROMPTS = []

for i in tqdm(range(PARAMS['num_samples'])):
    USER_PROMPTS.append(PARAMS['prompt_format'].format(user_p=PARAMS['user_prompt'], q=dataset_df['question'][i]))

gc.collect()

100%|██████████| 2000/2000 [00:00<00:00, 210536.29it/s]


40

In [6]:
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{USER_PROPMTS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(USER_PROMPTS, ensure_ascii=False, indent=1))

# сохраняем конфигурацию эксперимента
with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{PARAMS_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(PARAMS, ensure_ascii=False, indent=1))

In [7]:
print(USER_PROMPTS[0])

Generate an answer to the given question. If you do not have enough knowledge to generate correct answer based on them, then generate the following text: "I do not have an answer to your question". Generate answer in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.

Question:
When did Beyonce start becoming popular?

Answer:



### Генерируем ответы на вопросы

In [8]:
generate_answers, calc_metrics = [], []
display_iter = 100
s_time = time()
for i in tqdm(range(len(USER_PROMPTS))):
    pred_answer, meta_info = agent.generate(user_prompt=USER_PROMPTS[i], system_prompt=PARAMS['system_prompt'], gen_strategy=PARAMS['gen_strat'])

    cur_metrics = dict()
    if PARAMS['calculate_entropy']:
        logits = torch.cat(meta_info['logits'], 0).cpu().detach()
        entropy = compute_predictive_entropy(logits)
        cur_metrics['predictive_entropy'] = float(entropy)
    calc_metrics.append(cur_metrics)
    generate_answers.append(pred_answer)
    
    # logits = torch.cat(meta_info['logits'], 0).cpu().detach().numpy()
    # logits_int8 = logits.astype('int8') 
    # token_logits = {f"token_{i}": token_logits for i, token_logits in enumerate(logits_int8)}
    # pa_table = pa.table(token_logits)
    # pa.parquet.write_table(pa_table, f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{META_INFO_DIR_NAME}/logits_{i}.parquet")
    
    if i % display_iter == 0:
        print(f"\n[{i}]: \nGEN: {pred_answer}\nGOLD: {dataset_df['answer'][i]}\nMETRICS: {cur_metrics}")
e_time = time()

  0%|          | 1/2000 [00:00<29:16,  1.14it/s]


[0]: 
GEN: Beyoncé's debut album "Dangerously in Love" was released in 2003, which marked the beginning of her rise to fame.
GOLD: in the late 1990s
METRICS: {'predictive_entropy': 11.314181327819824}


  5%|▌         | 102/2000 [00:35<06:19,  5.00it/s]


[100]: 
GEN: 11 weeks
GOLD: eleven
METRICS: {'predictive_entropy': 4.589195251464844}


 10%|█         | 202/2000 [01:04<09:33,  3.13it/s]

...

 90%|█████████ | 1802/2000 [09:41<00:39,  5.05it/s]


[1800]: 
GEN: The CD player.
GOLD: IXI
METRICS: {'predictive_entropy': 6.282034873962402}


 95%|█████████▌| 1902/2000 [10:10<00:14,  6.75it/s]


[1900]: 
GEN: 2006
GOLD: September 12, 2006
METRICS: {'predictive_entropy': 2.3163957595825195}


100%|██████████| 2000/2000 [10:40<00:00,  3.12it/s]


In [9]:
# сохраняем используемые контексты + сгнерированные ответы
gen_info = []
for i in range(PARAMS['num_samples']):
    formated_contexts = []
    cur_item = {
        'gen_answer': str(generate_answers[i]), 
        'metainfo': calc_metrics[i], 
        'used_contexts': formated_contexts}
    gen_info.append(cur_item)

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{GEN_ANSW_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(gen_info, ensure_ascii=False, indent=1))

with open(f"{LOGS_SAVE_DIR}/v{PARAMS['version']}/{METADATA_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'elapsed_time': e_time - s_time}, ensure_ascii=False, indent=1))

### Оцениваем качество

In [10]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [11]:
LOADING_VERSION = "1"

In [12]:
with open(f'{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{GEN_ANSW_SAVE_NAME}','r', encoding='utf8') as fd:
    predicted_answers = list(map(lambda v: v['gen_answer'], json.loads(fd.read())))

In [13]:
metrics = ReaderMetrics(base_dir=BASE_DIR, model_path='ru_electra_medium')

Loading Meteor...
Loading ExactMatch


In [14]:
dataset_df = pd.read_csv(QA_DATASET_PATH)

In [15]:
target_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

stub_scores = {
    'BLEU2': [], 'BLEU1': [],
    'ExactMatch': [],'METEOR': [],
    'BertScore': [],
    'Levenshtain': [],
    'ROUGEL': []}

show_step = 10

process = tqdm(range(PARAMS['num_samples']))
target_answers =  dataset_df['answer'].to_list()[:PARAMS['num_samples']]
tmp_stub_pred_answers = []
for i in process:
    
    predicted_answer = predicted_answers[i]
    target_answer = target_answers[i]

    target_scores['BLEU1'] += metrics.bleu1([predicted_answer], [target_answer])
    target_scores['BLEU2'] += metrics.bleu2([predicted_answer], [target_answer])
    target_scores['ExactMatch'] += metrics.exact_match([predicted_answer], [target_answer])
    target_scores['METEOR'] += metrics.meteor([predicted_answer], [target_answer])
    target_scores['Levenshtain'] += metrics.levenshtain_score([predicted_answer], [target_answer])
    target_scores['ROUGEL'] += metrics.rougel([predicted_answer], [target_answer])


    stub_pred_answer = predicted_answer
    tmp_stub_pred_answers.append(stub_pred_answer)

    stub_scores['BLEU1'] += metrics.bleu1([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['BLEU2'] += metrics.bleu2([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ExactMatch'] += metrics.exact_match([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['METEOR'] += metrics.meteor([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['Levenshtain'] += metrics.levenshtain_score([stub_pred_answer], [PARAMS['stub_answer']])
    stub_scores['ROUGEL'] += metrics.rougel([stub_pred_answer], [PARAMS['stub_answer']])
            
    if i % show_step == 0:
        process.set_postfix({m_name: np.mean(score) for m_name, score in stub_scores.items()})

target_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in target_scores.items()}
target_scores['BertScore'] = metrics.bertscore(predicted_answers, target_answers)

stub_scores = {m_name: round(float(np.mean(score)), 5) for m_name, score in stub_scores.items()}
stub_scores['BertScore'] = metrics.bertscore(tmp_stub_pred_answers, [PARAMS['stub_answer']]*len(tmp_stub_pred_answers))
stub_scores['elapsed_time_sec'] = round(float(process.format_dict["elapsed"]), 3)

/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/jovyan/work/alexander_workspace/conda/rag_alex/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 2000/2000 [05:00<00:00,  6.65it/s, BLEU2=0.0463, BLEU1=0.0563, ExactMatch=0.0482, METEOR=0.0661, BertScore=nan, Levenshtain=46.2, ROUGEL=0.0656]


In [16]:
with open(f"{LOGS_SAVE_DIR}/v{LOADING_VERSION}/{SCORES_SAVE_NAME}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps({'target_answers': target_scores, 'stub_answers': stub_scores}, ensure_ascii=False, indent=1))